# 10 — DPO: sequence log-probs, a preference loss, and checking a paper's theory

**Paper:** Rafailov et al. (2023), *Direct Preference Optimization: Your Language Model is Secretly a Reward Model*: Eq. 1 (Bradley–Terry), Eq. 4 (optimal policy), Eq. 5 (implicit reward), Eq. 7 (loss), and §4 (gradient analysis).

**You will learn**
- $\log \pi(y|x)$ for a sequence: the **shift-by-one** of causal LMs, and masking out prompt/padding tokens
- numerically stable $\log\sigma(\cdot)$
- checking the paper's gradient analysis with autograd
- checking the paper's **main theorem** empirically: training with DPO reaches the closed-form optimal policy of Eq. 4

In [ ]:
import math
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from p2t import check, check_grad, seed

seed(0)

## 1. Sequence log-probability

Everything in RLHF and DPO needs $\log \pi_\theta(y\mid x)$, the log-probability of a whole response. By the chain rule of probability:
$$\log\pi(y\mid x) = \sum_{t=1}^{|y|}\log\pi(y_t \mid x, y_{<t})$$

**Decode it into a real LM batch.** This is where the paper's notation hides a lot:
- The model sees one concatenated sequence `[prompt tokens, response tokens, padding]` of shape `(B, T)`.
- A causal LM's `logits[:, i]` is the prediction for token `i + 1`. So you compare `logits[:, :-1]` against `labels[:, 1:]`. This is **the shift**.
- The sum runs only over **response** tokens. `mask` marks which *label* positions to count: 1 for response tokens, 0 for prompt and padding.

### Exercise 1

In [ ]:
def sequence_logprob(logits, labels, mask):
    """logits: (B, T, V), labels: (B, T) long, mask: (B, T) {0,1} marking tokens (in `labels`) to score.
    Returns (B,) sum of log p(labels[:, i] | prefix) over masked positions, using logits[:, i-1]."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
B, T, V = 3, 7, 11
logits = torch.randn(B, T, V)
labels = torch.randint(0, V, (B, T))
mask = torch.tensor([[0, 0, 1, 1, 1, 1, 0],   # prompt of 2, response of 4, 1 pad
                     [0, 0, 0, 1, 1, 1, 1],
                     [0, 1, 1, 0, 0, 0, 0]])

def seq_logprob_loops(logits, labels, mask):
    out = torch.zeros(B)
    for b in range(B):
        for i in range(1, T):
            if mask[b, i]:
                out[b] += torch.log(torch.softmax(logits[b, i - 1], -1)[labels[b, i]])
    return out

check("sequence_logprob", sequence_logprob(logits, labels, mask), seq_logprob_loops(logits, labels, mask))

## 2. The DPO loss

Eq. 7:
$$\mathcal{L}_\text{DPO}(\pi_\theta;\pi_\text{ref}) = -\mathbb{E}_{(x,y_w,y_l)\sim\mathcal{D}}\left[\log\sigma\!\left(\beta\log\frac{\pi_\theta(y_w|x)}{\pi_\text{ref}(y_w|x)} - \beta\log\frac{\pi_\theta(y_l|x)}{\pi_\text{ref}(y_l|x)}\right)\right]$$

**Decode it:**
- $y_w$ is the preferred ("winning", *chosen*) response and $y_l$ is the *rejected* one.
- Each ratio becomes a **difference of log-probs**: $\log\frac{\pi_\theta}{\pi_\text{ref}} = \log\pi_\theta - \log\pi_\text{ref}$. Never divide probabilities.
- $\hat r(x,y) = \beta\log\frac{\pi_\theta(y|x)}{\pi_\text{ref}(y|x)}$ is the **implicit reward** (Eq. 5, §5). Return it too, since it's what people log during training.
- $\log\sigma(z)$ computed naively is `-inf` for $z \lesssim -90$ in float32. Use `F.logsigmoid`, which computes $-\mathrm{softplus}(-z)$ stably.

Return **per-example** losses of shape `(B,)`. The caller takes the mean (or a weighted mean, which you'll need in §4).

### Exercise 2

In [ ]:
def dpo_loss(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta=0.1):
    """All inputs (B,) sequence log-probs. Returns (losses (B,), chosen_rewards (B,), rejected_rewards (B,))."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
pc, pr, rc, rr = (torch.randn(8) * 5 - 20 for _ in range(4))
beta = 0.1
naive = -torch.log(torch.sigmoid(beta * (pc - rc) - beta * (pr - rr)))
losses, cr, rj = dpo_loss(pc, pr, rc, rr, beta)
check("dpo_loss", losses, naive)
check("implicit rewards", cr, beta * (pc - rc))
check("policy == reference -> loss = log 2", dpo_loss(pc, pr, pc, pr, beta)[0], torch.full((8,), math.log(2)))
extreme = dpo_loss(torch.tensor([-5000.0]), torch.tensor([0.0]), torch.tensor([0.0]), torch.tensor([0.0]), beta=1.0)[0]
check("stable for a hugely wrong margin", extreme, torch.tensor([5000.0]))

### Exercise 3 — check the paper's gradient analysis

§4, "What does the DPO update do?":
$$\nabla_\theta\mathcal{L}_\text{DPO} = -\beta\,\mathbb{E}\Big[\underbrace{\sigma\big(\hat r(x,y_l) - \hat r(x,y_w)\big)}_{\text{higher weight when the implicit reward is wrong}}\big[\nabla_\theta\log\pi(y_w|x) - \nabla_\theta\log\pi(y_l|x)\big]\Big]$$

So the gradient of the **mean** loss with respect to `pi_chosen[i]` should be $-\frac{\beta}{B}\,\sigma(\hat r_l - \hat r_w)$, and with respect to `pi_rejected[i]` it should be the negative of that. Compute these analytically and compare with autograd.

In [ ]:
def dpo_grad_analytic(pi_chosen, pi_rejected, ref_chosen, ref_rejected, beta):
    """Return (d mean_loss / d pi_chosen, d mean_loss / d pi_rejected), each (B,)."""
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
pc_ = pc.clone().requires_grad_(); pr_ = pr.clone().requires_grad_()
dpo_loss(pc_, pr_, rc, rr, beta)[0].mean().backward()
g_w, g_l = dpo_grad_analytic(pc, pr, rc, rr, beta)
check("analytic grad wrt chosen logp", g_w, pc_.grad)
check("analytic grad wrt rejected logp", g_l, pr_.grad)

## 3. Experiment — does DPO actually find the optimal policy?

The paper's derivation starts from the KL-constrained RL objective, whose optimum is (Eq. 4):
$$\pi^*(y|x) = \frac{1}{Z(x)}\,\pi_\text{ref}(y|x)\exp\!\left(\frac{1}{\beta}r(x,y)\right)$$
The main claim is that minimizing $\mathcal{L}_\text{DPO}$ on Bradley–Terry preferences (Eq. 1: $p(y_1\succ y_2) = \sigma(r(y_1) - r(y_2))$) reaches this $\pi^*$ **without ever training a reward model**.

Test it on the smallest possible case: one prompt, $K = 5$ possible responses, and a policy given by a vector of logits. With a known true reward $r$ you can compute $\pi^*$ exactly, run DPO, and compare.

The dataset has *every* ordered pair $(i, j)$, weighted by the Bradley–Terry probability that $i$ is preferred, so the expected loss is computed exactly with no sampling noise.

### Exercise 4 — train the toy policy with your `dpo_loss`

In [ ]:
def train_toy_dpo(r_true, ref_logits, beta, steps=3000, lr=0.05):
    K = len(r_true)
    ref_logp = F.log_softmax(ref_logits, -1)
    wi, li = zip(*[(i, j) for i in range(K) for j in range(K) if i != j])
    wi, li = torch.tensor(wi), torch.tensor(li)
    pair_weight = torch.sigmoid(r_true[wi] - r_true[li])   # Bradley–Terry: P(w preferred over l)
    policy_logits = ref_logits.clone().requires_grad_()    # initialize the policy at the reference
    opt = torch.optim.Adam([policy_logits], lr=lr)
    # YOUR CODE HERE
    raise NotImplementedError
    return torch.softmax(policy_logits.detach(), -1)

In [ ]:
r_true = torch.tensor([2.0, 1.0, 0.0, 0.0, -1.0])
ref_logits = torch.tensor([0.0, 1.0, 0.5, -0.5, 1.5])
pi_ref = torch.softmax(ref_logits, -1)
results = {}
for beta in [0.5, 1.0, 4.0]:
    pi_star = pi_ref * torch.exp(r_true / beta); pi_star /= pi_star.sum()     # Eq. 4
    pi_dpo = train_toy_dpo(r_true, ref_logits, beta)
    results[beta] = (pi_star, pi_dpo)
    check(f"β={beta}: DPO policy == closed-form optimum (Eq. 4)", pi_dpo, pi_star, atol=2e-3)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5), sharey=True)
for a, (beta, (ps, pd)) in zip(ax, results.items()):
    xs = torch.arange(5)
    a.bar(xs - 0.25, pi_ref, 0.25, label="π_ref"); a.bar(xs, ps, 0.25, label="π* (Eq. 4)")
    a.bar(xs + 0.25, pd, 0.25, label="π_DPO"); a.set_title(f"β = {beta}"); a.set_xlabel("response")
ax[0].legend(); plt.show()

**Interpret the plot:** small $\beta$ lets the policy move far from the reference toward the high-reward response, and large $\beta$ keeps it close. $\beta$ is the strength of the KL penalty in the original RL objective, expressed here through the loss alone.

## Reflection
1. Why do you need $\pi_\text{ref}$ at all? What happens to Eq. 7 if you drop it, and how does the toy experiment's answer change?
2. In `sequence_logprob`, what goes wrong if you forget the shift? What if you forget to mask the prompt?
3. The analytic gradient weight $\sigma(\hat r_l - \hat r_w)$ goes to 0 when the model already ranks the pair correctly by a wide margin. Why is that a desirable property? Compare it to the unweighted version the paper ablates.